## ℹ️ Sobre este Notebook

Este notebook introduz o **Structured Streaming** do Spark. Explica o conceito de stream processing, simula um stream a partir de ficheiros, calcula métricas em janelas temporais e escreve resultados para sinks.


# Exemplo 11: Structured Streaming

**Objetivos:**
- Entender o conceito de stream processing
- Simular um stream a partir de ficheiros
- Calcular métricas em janelas temporais
- Escrever resultados para sink

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, window, count, avg, sum, max
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType
import time
import os

# Criar sessão Spark
spark = SparkSession.builder \
    .appName("Streaming_Demo") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/18 21:42:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.1


## 1. Definir Schema dos Dados

In [2]:
# Schema para dados de trading
trade_schema = StructType([
    StructField("timestamp", TimestampType(), True),
    StructField("symbol", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("volume", DoubleType(), True),
    StructField("exchange", StringType(), True)
])

print("Schema definido:")
for field in trade_schema.fields:
    print(f"  {field.name}: {field.dataType}")

Schema definido:
  timestamp: TimestampType()
  symbol: StringType()
  price: DoubleType()
  volume: DoubleType()
  exchange: StringType()


## 2. Gerar Dados de Exemplo

Vamos criar dados simulados de transações (trades) em streaming.

In [3]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Criar diretório para dados
input_dir = "stream_input"
os.makedirs(input_dir, exist_ok=True)

# Gerar dados simulados (divididos em batches)
np.random.seed(42)
symbols = ['BTC', 'ETH', 'SOL', 'ADA']
exchanges = ['Binance', 'Kraken', 'Coinbase']

base_time = datetime(2024, 1, 1, 9, 0, 0)

for batch in range(5):
    n_records = 100
    batch_time = base_time + timedelta(minutes=batch*5)
    
    data = {
        'timestamp': [batch_time + timedelta(seconds=int(x)) for x in np.random.randint(0, 300, n_records)],
        'symbol': np.random.choice(symbols, n_records),
        'price': np.random.uniform(100, 50000, n_records),
        'volume': np.random.uniform(0.1, 100, n_records),
        'exchange': np.random.choice(exchanges, n_records)
    }
    
    df_batch = pd.DataFrame(data)
    df_batch.to_json(f"{input_dir}/batch_{batch}.json", orient="records", lines=True)
    print(f"Criado batch_{batch}.json com {n_records} registos")

print(f"\nDados gerados em {input_dir}/")

Criado batch_0.json com 100 registos
Criado batch_1.json com 100 registos
Criado batch_2.json com 100 registos
Criado batch_3.json com 100 registos
Criado batch_4.json com 100 registos

Dados gerados em stream_input/


/tmp/ipykernel_1140249/819000710.py:29: Pandas4Warning: The default 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  df_batch.to_json(f"{input_dir}/batch_{batch}.json", orient="records", lines=True)
/tmp/ipykernel_1140249/819000710.py:29: Pandas4Warning: The default 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  df_batch.to_json(f"{input_dir}/batch_{batch}.json", orient="records", lines=True)
/tmp/ipykernel_1140249/819000710.py:29: Pandas4Warning: The default 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  df_batch.to_json(f"{input_dir}/batch_{batch}.json", orient="records", lines=True)
/tmp/ipykernel_1140249/819000710.py:29: Pandas4Warning: The default 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  df_batch.to_json(f"{in

## 3. Ler Stream de Dados

In [4]:
# Criar stream a partir de ficheiros JSON
stream_df = spark.readStream \
    .schema(trade_schema) \
    .json(input_dir)

print("Stream criado!")
print(f"Is streaming: {stream_df.isStreaming}")
print("\nSchema:")
stream_df.printSchema()

Stream criado!
Is streaming: True

Schema:
root
 |-- timestamp: timestamp (nullable = true)
 |-- symbol: string (nullable = true)
 |-- price: double (nullable = true)
 |-- volume: double (nullable = true)
 |-- exchange: string (nullable = true)



## 4. Transformações em Streaming

In [5]:
# Agregação simples: contagem por symbol
symbol_counts = stream_df.groupBy("symbol").count()

# Escrever para console (para debug)
query_console = symbol_counts.writeStream \
    .outputMode("complete") \
    .format("console") \
    .queryName("symbol_counts") \
    .start()

print("Query iniciada. Aguardando dados...")
time.sleep(10)  # Deixar processar
query_console.stop()
print("Query parada.")

26/04/18 21:42:49 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-0aab6390-c6e1-4f3d-9880-ba85a65f6281. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/18 21:42:49 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/04/18 21:42:49 WARN MicroBatchExecution: Disabling AQE since AQE is not supported in stateful workloads.


Query iniciada. Aguardando dados...


-------------------------------------------
Batch: 0
-------------------------------------------
+------+-----+
|symbol|count|
+------+-----+
|   ETH|  126|
|   BTC|  142|
|   SOL|  121|
|   ADA|  111|
+------+-----+

Query parada.


26/04/18 21:42:59 WARN DAGScheduler: Failed to cancel job group eaa88bdb-f1cf-46de-ab82-11fa20de7921. Cannot find active jobs for it.
26/04/18 21:42:59 WARN DAGScheduler: Failed to cancel job group eaa88bdb-f1cf-46de-ab82-11fa20de7921. Cannot find active jobs for it.


## 5. Agregações em Janelas Temporais

In [6]:
# Tumbling Window: janelas de 5 minutos sem overlap
windowed_stats = stream_df \
    .withWatermark("timestamp", "10 minutes") \
    .groupBy(
        window("timestamp", "5 minutes"),
        "symbol"
    ) \
    .agg(
        count("*").alias("n_trades"),
        avg("price").alias("avg_price"),
        sum("volume").alias("total_volume"),
        max("price").alias("max_price")
    )

# Mostrar resultado
query_window = windowed_stats.writeStream \
    .outputMode("append") \
    .format("console") \
    .option("truncate", "false") \
    .start()

time.sleep(15)
query_window.stop()

26/04/18 21:43:00 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-08a3e3ec-8a6e-4be0-9864-b4b9bcf46da1. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/18 21:43:00 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/04/18 21:43:00 WARN MicroBatchExecution: Disabling AQE since AQE is not supported in stateful workloads.


-------------------------------------------
Batch: 0
-------------------------------------------
+------+------+--------+---------+------------+---------+
|window|symbol|n_trades|avg_price|total_volume|max_price|
+------+------+--------+---------+------------+---------+
+------+------+--------+---------+------------+---------+



-------------------------------------------
Batch: 1
-------------------------------------------
+----------------------------------------------+------+--------+----------------+-------------+----------------+
|window                                        |symbol|n_trades|avg_price       |total_volume |max_price       |
+----------------------------------------------+------+--------+----------------+-------------+----------------+
|{+55970-10-10 13:00:00, +55970-10-10 13:05:00}|BTC   |1       |352.5730339263  |79.5391008574|352.5730339263  |
|{+55970-10-14 04:45:00, +55970-10-14 04:50:00}|SOL   |1       |35696.1443356392|16.5101532133|35696.1443356392|
|{+55970-10-17 22:45:00, +55970-10-17 22:50:00}|SOL   |1       |5244.3779260859 |12.1218423273|5244.3779260859 |
|{+55970-10-17 20:30:00, +55970-10-17 20:35:00}|ETH   |1       |26169.4853459513|0.6077057678 |26169.4853459513|
|{+55970-10-15 14:20:00, +55970-10-15 14:25:00}|BTC   |1       |22811.6698833042|25.3396444867|22811.6698833042|

26/04/18 21:43:15 WARN DAGScheduler: Failed to cancel job group d4d6c828-3e25-44db-827c-ec453e528ea7. Cannot find active jobs for it.
26/04/18 21:43:15 WARN DAGScheduler: Failed to cancel job group d4d6c828-3e25-44db-827c-ec453e528ea7. Cannot find active jobs for it.


## 6. Calcular VWAP (Volume Weighted Average Price)

Métrica comum em trading: preço médio ponderado pelo volume.

In [7]:
from pyspark.sql.functions import expr

# Calcular VWAP por janela
vwap_calc = stream_df \
    .withColumn("price_volume", col("price") * col("volume")) \
    .withWatermark("timestamp", "10 minutes") \
    .groupBy(
        window("timestamp", "5 minutes"),
        "symbol"
    ) \
    .agg(
        sum("price_volume").alias("sum_pv"),
        sum("volume").alias("sum_volume"),
        count("*").alias("count")
    ) \
    .withColumn("vwap", col("sum_pv") / col("sum_volume"))

# Mostrar VWAP
query_vwap = vwap_calc.select("window", "symbol", "vwap", "count").writeStream \
    .outputMode("append") \
    .format("console") \
    .option("truncate", "false") \
    .start()

time.sleep(15)
query_vwap.stop()

26/04/18 21:43:15 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-786eb965-b872-4573-bff7-29b951877f73. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/18 21:43:15 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/04/18 21:43:15 WARN MicroBatchExecution: Disabling AQE since AQE is not supported in stateful workloads.


-------------------------------------------
Batch: 0
-------------------------------------------
+------+------+----+-----+
|window|symbol|vwap|count|
+------+------+----+-----+
+------+------+----+-----+



-------------------------------------------
Batch: 1
-------------------------------------------
+----------------------------------------------+------+----------------+-----+
|window                                        |symbol|vwap            |count|
+----------------------------------------------+------+----------------+-----+
|{+55970-10-10 13:00:00, +55970-10-10 13:05:00}|BTC   |352.5730339263  |1    |
|{+55970-10-14 04:45:00, +55970-10-14 04:50:00}|SOL   |35696.1443356392|1    |
|{+55970-10-17 22:45:00, +55970-10-17 22:50:00}|SOL   |5244.3779260859 |1    |
|{+55970-10-17 20:30:00, +55970-10-17 20:35:00}|ETH   |26169.4853459513|1    |
|{+55970-10-15 14:20:00, +55970-10-15 14:25:00}|BTC   |22811.6698833042|1    |
|{+55970-10-09 23:55:00, +55970-10-10 00:00:00}|ADA   |4605.459525715  |1    |
|{+55970-10-24 18:55:00, +55970-10-24 19:00:00}|BTC   |2017.47481322   |1    |
|{+55970-10-11 02:20:00, +55970-10-11 02:25:00}|SOL   |17119.2109174079|1    |
|{+55970-10-09 14:10:00, +55970-10

26/04/18 21:43:30 WARN DAGScheduler: Failed to cancel job group d95a9031-5219-4472-8d19-efe1ea41234c. Cannot find active jobs for it.
26/04/18 21:43:30 WARN DAGScheduler: Failed to cancel job group d95a9031-5219-4472-8d19-efe1ea41234c. Cannot find active jobs for it.


## 7. Output Modes

| Mode | Descrição | Use case |
|------|-----------|----------|
| **Append** | Apenas novas rows | Agregações com watermark |
| **Complete** | Tabela completa a cada trigger | Aggregações sem janela |
| **Update** | Rows modificadas desde último trigger | Map operations |

In [8]:
# Escrever stream para Parquet (sink persistente)
output_dir = "stream_output"

query_file = windowed_stats.writeStream \
    .outputMode("append") \
    .format("parquet") \
    .option("path", output_dir) \
    .option("checkpointLocation", "checkpoint") \
    .start()

print(f"Escrevendo para {output_dir}/")
time.sleep(20)
query_file.stop()
print("Stream para ficheiro parado.")

26/04/18 21:43:30 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/04/18 21:43:30 WARN MicroBatchExecution: Disabling AQE since AQE is not supported in stateful workloads.


Escrevendo para stream_output/


26/04/18 21:43:33 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/18 21:43:33 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/04/18 21:43:33 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/04/18 21:43:33 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/04/18 21:43:33 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/04/18 21:43:33 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/04/18 21:43:33 WARN MemoryManager: Total allocation exceeds 95.

Stream para ficheiro parado.


26/04/18 21:43:50 WARN DAGScheduler: Failed to cancel job group a0c4c8a7-d9d1-465a-8c57-b4d169dd43e4. Cannot find active jobs for it.
26/04/18 21:43:50 WARN DAGScheduler: Failed to cancel job group a0c4c8a7-d9d1-465a-8c57-b4d169dd43e4. Cannot find active jobs for it.


In [9]:
# Ler os dados escritos
if os.path.exists(output_dir):
    output_df = spark.read.parquet(output_dir)
    print(f"Total de registos escritos: {output_df.count()}")
    output_df.show(10, truncate=False)
else:
    print("Diretório de saída não encontrado")

Total de registos escritos: 480
+----------------------------------------------+------+--------+----------------+-------------+----------------+
|window                                        |symbol|n_trades|avg_price       |total_volume |max_price       |
+----------------------------------------------+------+--------+----------------+-------------+----------------+
|{+55970-10-23 05:40:00, +55970-10-23 05:45:00}|BTC   |1       |2179.477740059  |55.5529864353|2179.477740059  |
|{+55970-10-13 12:05:00, +55970-10-13 12:10:00}|SOL   |1       |8932.8231654097 |33.7267673916|8932.8231654097 |
|{+55970-10-21 12:00:00, +55970-10-21 12:05:00}|ADA   |1       |46554.3399057982|21.2832795068|46554.3399057982|
|{+55970-10-12 21:55:00, +55970-10-12 22:00:00}|ADA   |1       |825.7788168273  |23.7989564944|825.7788168273  |
|{+55970-10-09 04:10:00, +55970-10-09 04:15:00}|ADA   |1       |36150.3605515491|4.1688073517 |36150.3605515491|
|{+55970-10-22 16:40:00, +55970-10-22 16:45:00}|ETH   |1       |

## 8. Ler de Kafka (exemplo de código)

```python
# Ler stream de Kafka
kafka_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", "trades") \
    .load()

# Os dados Kafka têm schema fixo: key, value, topic, partition, offset, timestamp
# Normalmente value está em bytes e precisa de parse JSON
from pyspark.sql.functions import from_json, col

parsed = kafka_stream.select(
    from_json(col("value").cast("string"), trade_schema).alias("data")
).select("data.*")

# Escrever para Kafka
query_kafka = result_df.writeStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("topic", "output") \
    .option("checkpointLocation", "checkpoint") \
    .start()
```

## Resumo

### Conceitos-chave:
1. **Stream = Tabela em contínua append**
2. **Watermark**: lidar com eventos atrasados
3. **Window functions**: agregações temporais
4. **Checkpoints**: tolerância a falhas

### Aplicações:
- Dashboards em tempo real
- Deteção de anomalias
- Métricas de trading (VWAP, volume)
- ETL contínuo

In [10]:
# Limpar
import shutil
for d in [input_dir, output_dir, "checkpoint"]:
    if os.path.exists(d):
        shutil.rmtree(d)
        
spark.stop()
print("Cleanup completo!")

Cleanup completo!


26/04/18 21:43:51 WARN StateStore: Error running maintenance thread
java.lang.IllegalStateException: SparkEnv not active, cannot do maintenance on StateStores
	at org.apache.spark.sql.execution.streaming.state.StateStore$.doMaintenance(StateStore.scala:1440)
	at org.apache.spark.sql.execution.streaming.state.StateStore$.$anonfun$startMaintenanceIfNeeded$1(StateStore.scala:1386)
	at org.apache.spark.sql.execution.streaming.state.StateStore$MaintenanceTask$$anon$1.run(StateStore.scala:1079)
	at java.base/java.util.concurrent.Executors$RunnableAdapter.call(Executors.java:539)
	at java.base/java.util.concurrent.FutureTask.runAndReset(FutureTask.java:305)
	at java.base/java.util.concurrent.ScheduledThreadPoolExecutor$ScheduledFutureTask.run(ScheduledThreadPoolExecutor.java:305)
	at java.base/java.util.concurrent.ThreadPoolExecutor.runWorker(ThreadPoolExecutor.java:1136)
	at java.base/java.util.concurrent.ThreadPoolExecutor$Worker.run(ThreadPoolExecutor.java:635)
	at java.base/java.lang.Thre